In [ ]:
# ============================================================
# Phase 3 - Imports and output paths
# ============================================================

import os
import re
import json
import time
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
from openai import OpenAI
import pprint as pp
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer, CrossEncoder
from helpers import load_corpus_txt

PHASE3_DIR = Path("outputs/phase3_outputs")
PHASE3_DIR.mkdir(exist_ok=True)

PHASE3_PLANS_PATH = PHASE3_DIR / "phase3_plans.json"
PHASE3_EVIDENCE_PATH = PHASE3_DIR / "phase3_subtopic_evidence.json"
PHASE3_REPORTS_PATH = PHASE3_DIR / "phase3_reports.json"
PHASE3_JUDGE_PATH = PHASE3_DIR / "phase3_judge_results.json"


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [2]:
# ============================================================
# Phase 3 - Configuration
# ============================================================

with open("outputs/phase1_outputs/retrieved_docs_knn.json", "r", encoding="utf-8") as f:
    phase1_knn_outputs = json.load(f)
  
docs = load_corpus_txt("./BioGen2024/filtered_pubmed_abstracts.txt")  
doc_by_pmid = {doc["pmid"]: doc for doc in docs}

BEST_QUERIES = {}

for qid, item in phase1_knn_outputs.items():
    BEST_QUERIES[qid] = item["query"]

# Number of subtopics generated by the planner
N_SUBTOPICS = 4

# Retrieval / evidence selection settings
PHASE3_TOP_K_DOCS = 3
PHASE3_TOP_SENTENCES_PER_DOC = 2

# Use all test queries by default
# For quick debugging, use: PHASE3_QIDS = list(BEST_QUERIES.keys())[:3]
PHASE3_QIDS = list(BEST_QUERIES.keys())

print("Number of Phase 3 queries:", len(PHASE3_QIDS))
print("Example QIDs:", PHASE3_QIDS[:5])

Number of Phase 3 queries: 33
Example QIDs: ['116', '118', '120', '122', '124']


In [ ]:
# ============================================================
# Phase 3 - LLM API clients
# ============================================================

# ---------------------------
# OpenSearch connection
# ---------------------------
OS_HOST = "api.novasearch.org"
OS_PORT = 443
OS_USER = "usernlp16"
OS_PASS = os.getenv("OS_PASS")

INDEX_NAME = OS_USER

# -------------------------------
# Generation model: Gemma4
# -------------------------------
GEN_BASE_URL = "https://api.novasearch.org/gemma4/v1"
GEN_API_KEY = os.getenv("GEN_API_KEY")
GEN_MODEL = "google/gemma-4-31B-it"

gen_client = OpenAI(
    base_url=GEN_BASE_URL,
    api_key=GEN_API_KEY
)

# -------------------------------
# Judge model: GPT-4o-mini
# Replace these with your IAedu API information
# -------------------------------
JUDGE_BASE_URL = os.getenv("IAEDU_API_URL")
JUDGE_API_KEY = os.getenv("IAEDU_API_KEY")
JUDGE_MODEL = "gpt-4o-mini"

judge_client = OpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=JUDGE_API_KEY
)

print("Generation model:", GEN_MODEL)
print("Judge model:", JUDGE_MODEL)

Generation model: google/gemma-4-31B-it
Judge model: gpt-4o-mini


In [4]:
# ============================================================
# Phase 3 - LLM helper functions
# ============================================================

def call_llm(client, model, messages, temperature=0.0, max_tokens=800, retries=3, sleep_time=5):
    """
    Generic chat completion call with retry logic.
    """
    last_error = None
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content.strip()
        
        except Exception as e:
            last_error = e
            print(f"API error on attempt {attempt + 1}/{retries}: {e}")
            time.sleep(sleep_time)
    
    raise RuntimeError(f"LLM call failed after {retries} attempts. Last error: {last_error}")


def extract_json_from_text(text):
    """
    Extract JSON from a model response.
    Handles cases where the model wraps JSON in markdown.
    """
    text = text.strip()
    
    # Remove markdown fences if present
    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()
    
    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Try extracting first JSON array or object
    array_match = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if array_match:
        return json.loads(array_match.group(0))
    
    object_match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if object_match:
        return json.loads(object_match.group(0))
    
    raise ValueError("Could not extract valid JSON from response:\n" + text)

In [5]:
# ============================================================
# Phase 3 - Planner
# ============================================================

def build_planner_prompt(question, n_subtopics=4):
    return f"""
You are a biomedical research planner.

Given a patient biomedical question, decompose it into {n_subtopics} focused subtopics.
Each subtopic should help retrieve PubMed evidence and write a structured deep research report.

Rules:
- The subtopics must be medically relevant.
- Avoid duplicate subtopics.
- Each subtopic must include a search query suitable for PubMed/OpenSearch retrieval.
- Return valid JSON only.
- Do not include explanations outside the JSON.

Return format:
[
  {{
    "title": "short subtopic title",
    "search_query": "search query for retrieval",
    "rationale": "why this subtopic is useful"
  }}
]

Patient question:
{question}
""".strip()


def plan_subtopics(question, n_subtopics=4):
    prompt = build_planner_prompt(question, n_subtopics=n_subtopics)
    
    messages = [
        {
            "role": "system",
            "content": "You are a biomedical research planning assistant."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    response = call_llm(
        client=gen_client,
        model=GEN_MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=900
    )
    
    subtopics = extract_json_from_text(response)
    
    # Safety cleanup
    cleaned = []
    for i, subtopic in enumerate(subtopics[:n_subtopics], start=1):
        cleaned.append({
            "subtopic_id": i,
            "title": subtopic.get("title", f"Subtopic {i}"),
            "search_query": subtopic.get("search_query", question),
            "rationale": subtopic.get("rationale", "")
        })
    
    return cleaned

In [6]:
# ============================================================
# Phase 3 - Run or load planning
# ============================================================

if PHASE3_PLANS_PATH.exists():
    print("Loading existing plans from:", PHASE3_PLANS_PATH)
    
    with open(PHASE3_PLANS_PATH, "r", encoding="utf-8") as f:
        phase3_plans = json.load(f)

else:
    phase3_plans = {}
    
    for qid in tqdm(PHASE3_QIDS, desc="Planning subtopics"):
        question = BEST_QUERIES[qid]
        
        phase3_plans[qid] = {
            "qid": qid,
            "question": question,
            "subtopics": plan_subtopics(
                question=question,
                n_subtopics=N_SUBTOPICS
            )
        }
    
    with open(PHASE3_PLANS_PATH, "w", encoding="utf-8") as f:
        json.dump(phase3_plans, f, indent=2, ensure_ascii=False)
    
    print("Saved:", PHASE3_PLANS_PATH)

print("Number of plans:", len(phase3_plans))

# Preview one plan
first_qid = next(iter(phase3_plans))
phase3_plans[first_qid]

Loading existing plans from: phase3_outputs\phase3_plans.json
Number of plans: 33


{'qid': '116',
 'question': 'natural treatments for sleep apnea Are there ways to prevent sleep apnea or treat it naturally? The patient is looking for natural remedies to prevent and treat sleep apnea.',
 'subtopics': [{'subtopic_id': 1,
   'title': 'Lifestyle Interventions and Weight Management',
   'search_query': '("obstructive sleep apnea" OR "OSA") AND ("weight loss" OR "diet" OR "exercise" OR "lifestyle modification")',
   'rationale': 'Weight reduction is a primary non-pharmacological treatment for OSA as it reduces upper airway collapse.'},
  {'subtopic_id': 2,
   'title': 'Positional Therapy and Sleep Hygiene',
   'search_query': '("obstructive sleep apnea" OR "OSA") AND ("positional therapy" OR "side sleeping" OR "sleep hygiene")',
   'rationale': 'Many patients experience apnea primarily when supine; identifying positional treatments provides a natural way to reduce events.'},
  {'subtopic_id': 3,
   'title': 'Myofunctional Therapy and Oral Exercises',
   'search_query': '(

In [7]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Connect to OpenSearch

In [8]:
client = OpenSearch(
    hosts = [{'host': OS_HOST, 'port': OS_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OS_USER, OS_PASS),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

if client.indices.exists(index=INDEX_NAME):

    resp = client.indices.open(index=INDEX_NAME)
    print(resp)

    print('\n----------------------------------------------------------------------------------- INDEX SETTINGS')
    settings = client.indices.get_settings(index=INDEX_NAME)
    pp.pprint(settings)

    print('\n----------------------------------------------------------------------------------- INDEX MAPPINGS')
    mappings = client.indices.get_mapping(index=INDEX_NAME)
    pp.pprint(mappings)

    print('\n----------------------------------------------------------------------------------- INDEX #DOCs')
    print(client.count(index=INDEX_NAME))
else:
    print("Index does not exist.")

{'acknowledged': True, 'shards_acknowledged': True}

----------------------------------------------------------------------------------- INDEX SETTINGS
{'usernlp16': {'settings': {'index': {'creation_date': '1777384806261',
                                      'knn': 'true',
                                      'knn.derived_source': {'enabled': 'true'},
                                      'number_of_replicas': '0',
                                      'number_of_shards': '4',
                                      'provided_name': 'usernlp16',
                                      'refresh_interval': '-1',
                                      'replication': {'type': 'DOCUMENT'},
                                      'similarity': {'dirichlet_similarity': {'mu': '150',
                                                                              'type': 'LMDirichlet'},
                                                     'jm_similarity': {'lambda': '0.5',
                          

In [9]:
# ============================================================
# Phase 3 - KNN retrieval for subtopics
# Based on the Phase 1 KNN implementation
# ============================================================

def retrieve_knn_documents(query_text, top_k=3):
    """
    Retrieve documents from OpenSearch using the same KNN setup used in Phase 1.

    Phase 1 setup:
    - OpenSearch client: client
    - Index name: INDEX_NAME
    - Embedding model: embedder
    - Vector field: embedding
    - Indexed source fields: pmid, title, abstract
    """
    
    query_vector = embedder.encode(query_text).tolist()

    response = client.search(
        index=INDEX_NAME,
        body={
            "size": top_k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": query_vector,
                        "k": top_k
                    }
                }
            },
            "_source": ["pmid", "title", "abstract"]
        }
    )

    retrieved_docs = []

    for rank, hit in enumerate(response["hits"]["hits"], start=1):
        source = hit["_source"]
        pmid = str(source.get("pmid", ""))

        # Prefer local parsed document if available
        doc = doc_by_pmid.get(pmid, {})

        retrieved_docs.append({
            "rank": rank,
            "doc_rank": rank,
            "pmid": pmid,
            "score": float(hit.get("_score", 0.0)),
            "retrieval_score": float(hit.get("_score", 0.0)),
            "title": doc.get("title", source.get("title", "")),
            "abstract": doc.get("abstract", source.get("abstract", ""))
        })

    return retrieved_docs

In [10]:
cross_encoder = CrossEncoder("ncbi/MedCPT-Cross-Encoder")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:
def split_abstract_into_sentences(text):
    if not text:
        return []

    text = re.sub(r"\s+", " ", text.strip())

    # Simple biomedical-safe sentence splitter
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # Remove very short or empty sentences
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]

    return sentences

def select_reference_sentences_for_query(
    qid,
    query_text,
    retrieved_docs,
    doc_by_pmid,
    top_k_docs=3,
    top_sentences_per_doc=3,
):
    query_results = []

    top_docs = retrieved_docs[:top_k_docs]

    for rank, item in enumerate(top_docs, start=1):
        pmid = str(item[0])
        retrieval_score = float(item[1])

        if pmid not in doc_by_pmid:
            continue

        doc = doc_by_pmid[pmid]
        title = doc.get("title", "")
        abstract = doc.get("abstract", "")

        sentences = split_abstract_into_sentences(abstract)

        if not sentences:
            continue

        pairs = [(query_text, sent) for sent in sentences]
        scores = cross_encoder.predict(pairs)

        scored_sentences = []
        for sent, score in zip(sentences, scores):
            scored_sentences.append({
                "sentence": sent,
                "cross_encoder_score": float(score)
            })

        scored_sentences = sorted(
            scored_sentences,
            key=lambda x: x["cross_encoder_score"],
            reverse=True
        )

        selected = scored_sentences[:top_sentences_per_doc]

        query_results.append({
            "qid": qid,
            "query": query_text,
            "pmid": pmid,
            "doc_rank": rank,
            "retrieval_score": retrieval_score,
            "title": title,
            "selected_sentences": selected
        })

    return query_results

In [12]:
# ============================================================
# Phase 3 - Explore subtopic
# Compatible with Phase 2 select_reference_sentences_for_query()
# ============================================================

def explore_subtopic(qid, subtopic):
    """
    Browse/explore one subtopic:
    1. Retrieve documents with KNN
    2. Convert retrieved docs to the Phase 1/2 run format: [(pmid, score), ...]
    3. Select reference sentences with the MedCPT cross-encoder
    """
    
    subtopic_query = subtopic["search_query"]
    
    retrieved_docs_full = retrieve_knn_documents(
        query_text=subtopic_query,
        top_k=PHASE3_TOP_K_DOCS
    )
    
    # Convert dict format to the format expected by your Phase 2 function
    retrieved_docs_for_phase2 = [
        (
            doc["pmid"],
            doc.get("retrieval_score", doc.get("score", 0.0))
        )
        for doc in retrieved_docs_full
    ]
    
    reference_sentences = select_reference_sentences_for_query(
        qid=f"{qid}_subtopic_{subtopic['subtopic_id']}",
        query_text=subtopic_query,
        retrieved_docs=retrieved_docs_for_phase2,
        doc_by_pmid=doc_by_pmid,
        top_k_docs=PHASE3_TOP_K_DOCS,
        top_sentences_per_doc=PHASE3_TOP_SENTENCES_PER_DOC
    )
    
    return {
        "subtopic_id": subtopic["subtopic_id"],
        "title": subtopic["title"],
        "search_query": subtopic_query,
        "rationale": subtopic.get("rationale", ""),
        "retrieved_docs": retrieved_docs_full,
        "reference_sentences": reference_sentences
    }

In [13]:
# ============================================================
# Phase 3 - Run or load subtopic exploration
# ============================================================

if PHASE3_EVIDENCE_PATH.exists():
    print("Loading existing Phase 3 evidence from:", PHASE3_EVIDENCE_PATH)
    
    with open(PHASE3_EVIDENCE_PATH, "r", encoding="utf-8") as f:
        phase3_evidence = json.load(f)

else:
    phase3_evidence = {}
    
    for qid, plan in tqdm(phase3_plans.items(), desc="Exploring subtopics"):
        phase3_evidence[qid] = {
            "qid": qid,
            "question": plan["question"],
            "subtopics": []
        }
        
        for subtopic in plan["subtopics"]:
            explored = explore_subtopic(qid, subtopic)
            phase3_evidence[qid]["subtopics"].append(explored)
    
    with open(PHASE3_EVIDENCE_PATH, "w", encoding="utf-8") as f:
        json.dump(phase3_evidence, f, indent=2, ensure_ascii=False)
    
    print("Saved:", PHASE3_EVIDENCE_PATH)

print("Number of explored queries:", len(phase3_evidence))

# Preview
first_qid = next(iter(phase3_evidence))
phase3_evidence[first_qid]

Loading existing Phase 3 evidence from: phase3_outputs\phase3_subtopic_evidence.json
Number of explored queries: 33


{'qid': '116',
 'question': 'natural treatments for sleep apnea Are there ways to prevent sleep apnea or treat it naturally? The patient is looking for natural remedies to prevent and treat sleep apnea.',
 'subtopics': [{'subtopic_id': 1,
   'title': 'Lifestyle Interventions and Weight Management',
   'search_query': '("obstructive sleep apnea" OR "OSA") AND ("weight loss" OR "diet" OR "exercise" OR "lifestyle modification")',
   'rationale': 'Weight reduction is a primary non-pharmacological treatment for OSA as it reduces upper airway collapse.',
   'retrieved_docs': [{'rank': 1,
     'doc_rank': 1,
     'pmid': '21603432',
     'score': 0.60276717,
     'retrieval_score': 0.60276717,
     'title': 'Obstructive sleep apnea: a growing problem.',
     'abstract': 'Obstructive sleep apnea is an underrecognized and underdiagnosed medical condition, with a myriad of negative consequences on patients\' health and society as a whole. Symptoms include daytime sleepiness, loud snoring, and re

In [14]:
# ============================================================
# Phase 3 - Evidence aggregation
# ============================================================

def flatten_reference_sentences(reference_sentences):
    """
    Converts the output of select_reference_sentences_for_query into a flat list.
    Expected structure:
    [
      {
        "pmid": "...",
        "title": "...",
        "selected_sentences": [
          {"sentence": "...", "cross_encoder_score": ...}
        ]
      }
    ]
    """
    flat = []
    
    for doc in reference_sentences:
        pmid = str(doc.get("pmid", ""))
        title = doc.get("title", "")
        doc_rank = doc.get("doc_rank", doc.get("rank", None))
        retrieval_score = doc.get("retrieval_score", doc.get("score", None))
        
        for sent_obj in doc.get("selected_sentences", []):
            sentence = sent_obj.get("sentence", "").strip()
            
            if not sentence:
                continue
            
            flat.append({
                "pmid": pmid,
                "title": title,
                "doc_rank": doc_rank,
                "retrieval_score": retrieval_score,
                "sentence": sentence,
                "cross_encoder_score": sent_obj.get("cross_encoder_score", None)
            })
    
    return flat


def aggregate_evidence_for_query(query_evidence, max_sentences_per_subtopic=6):
    """
    Deduplicate and organize evidence by subtopic.
    """
    aggregated = []
    seen = set()
    
    for subtopic in query_evidence["subtopics"]:
        flat_sentences = flatten_reference_sentences(subtopic["reference_sentences"])
        
        # Sort by cross-encoder score, highest first
        flat_sentences = sorted(
            flat_sentences,
            key=lambda x: x["cross_encoder_score"] if x["cross_encoder_score"] is not None else -999,
            reverse=True
        )
        
        selected = []
        
        for item in flat_sentences:
            key = (item["pmid"], item["sentence"])
            
            if key in seen:
                continue
            
            seen.add(key)
            selected.append(item)
            
            if len(selected) >= max_sentences_per_subtopic:
                break
        
        aggregated.append({
            "subtopic_id": subtopic["subtopic_id"],
            "title": subtopic["title"],
            "search_query": subtopic["search_query"],
            "rationale": subtopic.get("rationale", ""),
            "evidence": selected
        })
    
    return aggregated


def format_evidence_for_prompt(aggregated_evidence):
    """
    Format evidence into a readable text block for the report generator.
    """
    lines = []
    
    for subtopic in aggregated_evidence:
        lines.append(f"SUBTOPIC {subtopic['subtopic_id']}: {subtopic['title']}")
        lines.append(f"Search query: {subtopic['search_query']}")
        
        for i, ev in enumerate(subtopic["evidence"], start=1):
            lines.append(
                f"- Evidence {i}: PMID {ev['pmid']} | "
                f"Title: {ev['title']} | "
                f"Sentence: {ev['sentence']}"
            )
        
        lines.append("")
    
    return "\n".join(lines).strip()

In [15]:
# ============================================================
# Phase 3 - Final report generation
# ============================================================

def build_deep_research_report_prompt(question, evidence_text):
    return f"""
You are a biomedical deep research assistant.

Write a structured, patient-facing biomedical research report using ONLY the provided evidence.

Rules:
- Use only the evidence provided below.
- Do not add outside medical knowledge.
- Every factual claim must include a PMID citation.
- Use citations in this exact format: [PMID:123456].
- Do not cite PMIDs that are not present in the evidence.
- If evidence is weak, limited, indirect, or conflicting, explicitly say so.
- Do not diagnose the patient.
- Do not give unsafe instructions.
- Write clearly for a non-expert patient.

Required report structure:
1. Short answer
2. Evidence by subtopic
3. Limitations or uncertainty in the evidence
4. Practical interpretation
5. Conclusion

Patient question:
{question}

Evidence:
{evidence_text}

Final report:
""".strip()


def generate_deep_research_report(question, aggregated_evidence):
    evidence_text = format_evidence_for_prompt(aggregated_evidence)
    prompt = build_deep_research_report_prompt(question, evidence_text)
    
    messages = [
        {
            "role": "system",
            "content": "You are a careful biomedical research assistant that writes grounded reports with citations."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    report = call_llm(
        client=gen_client,
        model=GEN_MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=1400
    )
    
    return report.strip()

In [16]:
# ============================================================
# Phase 3 - Run or load final reports
# ============================================================

if PHASE3_REPORTS_PATH.exists():
    print("Loading existing Phase 3 reports from:", PHASE3_REPORTS_PATH)
    
    with open(PHASE3_REPORTS_PATH, "r", encoding="utf-8") as f:
        phase3_reports = json.load(f)

else:
    phase3_reports = {}
    
    for qid, query_evidence in tqdm(phase3_evidence.items(), desc="Generating deep research reports"):
        aggregated_evidence = aggregate_evidence_for_query(
            query_evidence,
            max_sentences_per_subtopic=6
        )
        
        report = generate_deep_research_report(
            question=query_evidence["question"],
            aggregated_evidence=aggregated_evidence
        )
        
        phase3_reports[qid] = {
            "qid": qid,
            "question": query_evidence["question"],
            "aggregated_evidence": aggregated_evidence,
            "report": report
        }
    
    with open(PHASE3_REPORTS_PATH, "w", encoding="utf-8") as f:
        json.dump(phase3_reports, f, indent=2, ensure_ascii=False)
    
    print("Saved:", PHASE3_REPORTS_PATH)

print("Number of reports:", len(phase3_reports))

# Preview one report
first_qid = next(iter(phase3_reports))
print("QID:", first_qid)
print(phase3_reports[first_qid]["report"])

Loading existing Phase 3 reports from: phase3_outputs\phase3_reports.json
Number of reports: 33
QID: 116
# Biomedical Research Report: Natural Approaches to Obstructive Sleep Apnea (OSA)

### 1. Short Answer
There are several natural and lifestyle-based strategies that may help manage or treat obstructive sleep apnea (OSA), including weight loss, avoiding sleeping on your back (positional therapy), and oropharyngeal exercises. While these complementary and behavioral approaches can be beneficial, conventional therapies like CPAP and oral appliances are noted as providing the best opportunity for symptomatic improvement [PMID: 36088151, PMID: 37532368].

### 2. Evidence by Subtopic

**Lifestyle Interventions and Weight Management**
*   **Weight and Habits:** Conservative approaches, specifically weight loss and the cessation of tobacco and alcohol use, are strongly encouraged for patients with OSA [PMID: 21603432].
*   **Health Outcomes:** Applying lifestyle modifications can reduce the

In [ ]:
# ============================================================
# Phase 3 - Citation validation
# ============================================================

def extract_pmids_from_text(text):
    """
    Extract PMIDs from flexible citation formats, including:
    [PMID: 123456]
    [PMID: 123456, PMID: 7891011]
    PMID: 123456
    PMID 123456
    [123456]
    """
    text = str(text)

    pmids = set()

    # Captures PMID: 123456 and PMID 123456 anywhere in the report
    for match in re.findall(r"PMID[:\s]+(\d{5,9})", text, flags=re.IGNORECASE):
        pmids.add(match)

    # Captures standalone numeric citations like [123456]
    for match in re.findall(r"\[(\d{5,9})\]", text):
        pmids.add(match)

    return sorted(pmids)


def get_valid_pmids_from_evidence(aggregated_evidence):
    valid_pmids = set()
    
    for subtopic in aggregated_evidence:
        for ev in subtopic["evidence"]:
            valid_pmids.add(str(ev["pmid"]))
    
    return sorted(valid_pmids)


def validate_report_citations(report, aggregated_evidence):
    cited_pmids = extract_pmids_from_text(report)
    valid_pmids = get_valid_pmids_from_evidence(aggregated_evidence)
    
    invalid_pmids = sorted(set(cited_pmids) - set(valid_pmids))
    missing = len(cited_pmids) == 0
    
    return {
        "cited_pmids": cited_pmids,
        "valid_pmids": valid_pmids,
        "invalid_pmids": invalid_pmids,
        "has_no_citations": missing,
        "all_citations_valid": len(invalid_pmids) == 0 and not missing
    }


# Run citation validation for all reports
for qid, item in phase3_reports.items():
    item["citation_validation"] = validate_report_citations(
        item["report"],
        item["aggregated_evidence"]
    )

# Show problems
for qid, item in phase3_reports.items():
    validation = item["citation_validation"]
    
    if not validation["all_citations_valid"]:
        print("Citation issue in QID:", qid)
        print(validation)
        
with open(PHASE3_REPORTS_PATH, "w", encoding="utf-8") as f:
    json.dump(phase3_reports, f, indent=2, ensure_ascii=False)

print("Citation validation added to reports and saved:", PHASE3_REPORTS_PATH)

Citation issue in QID: 116
{'cited_pmids': [], 'valid_pmids': ['19037617', '20875158', '21603432', '21642831', '22014867', '24118690', '28901030', '29517065', '30709524', '33659106', '36088151', '37532368'], 'invalid_pmids': [], 'has_no_citations': True, 'all_citations_valid': False}
Citation issue in QID: 118
{'cited_pmids': [], 'valid_pmids': ['15259527', '23317075', '23652196', '25140844', '27119941', '29217021', '29355667', '31598519'], 'invalid_pmids': [], 'has_no_citations': True, 'all_citations_valid': False}
Citation issue in QID: 120
{'cited_pmids': [], 'valid_pmids': ['17305614', '24053080', '31097197', '31471563', '31783736', '32445717', '33133816'], 'invalid_pmids': [], 'has_no_citations': True, 'all_citations_valid': False}
Citation issue in QID: 122
{'cited_pmids': [], 'valid_pmids': ['11728347', '15842434', '16308114', '18616439', '21140289', '2148032', '24276094', '32975832'], 'invalid_pmids': [], 'has_no_citations': True, 'all_citations_valid': False}
Citation issue in

In [18]:
# ============================================================
# IAedu GPT-4o-mini judge setup for Phase 3
# ============================================================

import os
import json
import ast
import uuid
import time
import requests

IAEDU_API_KEY = os.getenv("IAEDU_API_KEY")
IAEDU_API_URL = os.getenv("IAEDU_API_URL")

IAEDU_CHANNEL_ID = "cmoh3pa5o286kj301ga04tio8"
IAEDU_PHASE3_THREAD_ID = "judge-thread-phase3"

VALID_PHASE3_VALUES = {
    "factual_support": ["Supported", "Partially Supported", "Unsupported"],
    "citation_correctness": ["Correct", "Partially Correct", "Incorrect"],
    "completeness": ["Complete", "Partially Complete", "Incomplete"],
    "safety": ["Safe", "Potentially Unsafe"]
}

In [19]:
# ============================================================
# IAedu parser for Phase 3 JSON judge responses
# ============================================================

def _is_uuid(text):
    try:
        uuid.UUID(str(text).strip())
        return True
    except (ValueError, AttributeError):
        return False


def _is_rate_limit(text):
    low = str(text).lower()
    return "rate limit" in low or "429" in low or "too many requests" in low


def _extract_first_json_object(text):
    """
    Extracts the first valid JSON object from a messy string.
    Useful when IAedu returns: Processing{...}{...}uuid
    """
    text = str(text).strip()
    
    for start in [i for i, ch in enumerate(text) if ch == "{"]:
        depth = 0
        
        for end in range(start, len(text)):
            if text[end] == "{":
                depth += 1
            elif text[end] == "}":
                depth -= 1
            
            if depth == 0:
                candidate = text[start:end + 1]
                
                try:
                    return json.loads(candidate)
                except Exception:
                    try:
                        obj = ast.literal_eval(candidate)
                        if isinstance(obj, dict):
                            return obj
                    except Exception:
                        break
    
    return None


# ============================================================
# IAedu parser for token-streamed JSON responses
# ============================================================

def _parse_iaedu_json_response(raw_text):
    token_text = ""
    candidate_objects = []

    for raw_line in raw_text.splitlines():
        line = raw_line.strip()

        if line.startswith("data:"):
            line = line[len("data:"):].lstrip()

        if not line or line in {"[DONE]", "DONE", "Processing"}:
            continue

        if _is_uuid(line):
            continue

        obj = None
        try:
            obj = json.loads(line)
        except Exception:
            try:
                obj = ast.literal_eval(line)
            except Exception:
                obj = None

        if isinstance(obj, dict):
            event_type = obj.get("type")
            content = obj.get("content", "")

            if event_type == "error" or _is_rate_limit(content):
                raise RuntimeError(f"IAedu rate limit or stream error: {obj}")

            # Main fix: collect streamed tokens
            if event_type == "token":
                token_text += str(content)
                continue

            # Ignore lifecycle events
            if event_type in {"start", "done"}:
                continue

            # Sometimes the response is already the final JSON object
            if any(key in obj for key in VALID_PHASE3_VALUES.keys()):
                candidate_objects.append(obj)
                continue

            # Sometimes the final JSON is inside content
            if isinstance(content, str):
                embedded = _extract_first_json_object(content)
                if isinstance(embedded, dict) and any(
                    key in embedded for key in VALID_PHASE3_VALUES.keys()
                ):
                    candidate_objects.append(embedded)
                continue

        else:
            embedded = _extract_first_json_object(line)
            if isinstance(embedded, dict) and any(
                key in embedded for key in VALID_PHASE3_VALUES.keys()
            ):
                candidate_objects.append(embedded)

    # First try the reconstructed token stream
    if token_text.strip():
        obj = _extract_first_json_object(token_text)
        if isinstance(obj, dict) and any(
            key in obj for key in VALID_PHASE3_VALUES.keys()
        ):
            return obj

    # Then try candidate objects
    if candidate_objects:
        return candidate_objects[-1]

    # Final fallback: try entire raw response
    obj = _extract_first_json_object(raw_text)
    if isinstance(obj, dict) and any(
        key in obj for key in VALID_PHASE3_VALUES.keys()
    ):
        return obj

    raise RuntimeError(
        "Could not parse a valid Phase 3 judgment JSON from IAedu response.\n"
        f"Reconstructed token text:\n{token_text[:1000]}\n\n"
        f"Raw response:\n{raw_text[:1500]}"
    )

In [20]:
# ============================================================
# Phase 3 judgment normalization
# ============================================================

def _normalize_value(value, allowed_values, default):
    text = str(value).strip().replace('"', "").replace("'", "")
    
    for allowed in allowed_values:
        if text.lower() == allowed.lower():
            return allowed
    
    for allowed in allowed_values:
        if allowed.lower() in text.lower():
            return allowed
    
    return default


def normalize_phase3_judgment(obj):
    """
    Ensures the judge output has the expected keys and valid values.
    """
    if not isinstance(obj, dict):
        raise ValueError(f"Expected dict judgment, got: {type(obj)}")

    explanation = str(obj.get("short_explanation", "")).strip()

    if not explanation:
        explanation = "No explanation returned by the judge."

    return {
        "factual_support": _normalize_value(
            obj.get("factual_support", ""),
            VALID_PHASE3_VALUES["factual_support"],
            "Partially Supported"
        ),
        "citation_correctness": _normalize_value(
            obj.get("citation_correctness", ""),
            VALID_PHASE3_VALUES["citation_correctness"],
            "Partially Correct"
        ),
        "completeness": _normalize_value(
            obj.get("completeness", ""),
            VALID_PHASE3_VALUES["completeness"],
            "Partially Complete"
        ),
        "safety": _normalize_value(
            obj.get("safety", ""),
            VALID_PHASE3_VALUES["safety"],
            "Potentially Unsafe"
        ),
        "short_explanation": explanation
    }

In [21]:
# ============================================================
# Phase 3 IAedu judge call
# ============================================================

def call_iaedu_judge_json(prompt, max_retries=6):
    headers = {
        "x-api-key": IAEDU_API_KEY
    }
    
    data = {
        "channel_id": IAEDU_CHANNEL_ID,
        "thread_id": IAEDU_PHASE3_THREAD_ID,
        "user_info": "{}",
        "message": prompt
    }

    for attempt in range(max_retries):
        response = requests.post(
            IAEDU_API_URL,
            headers=headers,
            data=data,
            timeout=120,
            stream=True
        )

        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 0)) or (2 ** attempt) * 10
            print(f"[HTTP 429] Waiting {wait}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait)
            continue

        response.raise_for_status()

        raw_lines = [
            line for line in response.iter_lines(decode_unicode=True)
            if line is not None
        ]
        raw_text = "\n".join(raw_lines)

        if _is_rate_limit(raw_text):
            wait = (2 ** attempt) * 10
            print(f"[Body 429] Waiting {wait}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait)
            continue

        obj = _parse_iaedu_json_response(raw_text)
        return normalize_phase3_judgment(obj)

    raise RuntimeError(f"IAedu still rate-limiting after {max_retries} retries.")

In [22]:
test_prompt = """
Return exactly this JSON object and nothing else:

{
  "factual_support": "Supported",
  "citation_correctness": "Correct",
  "completeness": "Complete",
  "safety": "Safe",
  "short_explanation": "The report is fully supported by the provided CBD evidence and uses the correct PMID citation."
}
"""
test_judgment = call_iaedu_judge_json(test_prompt)
test_judgment

[Body 429] Waiting 10s (attempt 1/6)...


{'factual_support': 'Supported',
 'citation_correctness': 'Correct',
 'completeness': 'Complete',
 'safety': 'Safe',
 'short_explanation': 'The report is fully supported by the provided CBD evidence and uses the correct PMID citation.'}

In [23]:
# ============================================================
# Phase 3 - LLM-as-a-judge prompt
# ============================================================

def format_all_evidence_sentences(aggregated_evidence):
    lines = []
    
    for subtopic in aggregated_evidence:
        lines.append(f"Subtopic: {subtopic['title']}")
        
        for ev in subtopic["evidence"]:
            lines.append(
                f"[PMID:{ev['pmid']}] {ev['sentence']}"
            )
        
        lines.append("")
    
    return "\n".join(lines).strip()


def build_phase3_judge_prompt(question, report, aggregated_evidence):
    evidence_text = format_all_evidence_sentences(aggregated_evidence)
    
    return f"""
You are an expert biomedical evaluator.

Evaluate the final deep research report using only the provided evidence.

You must judge four criteria:

1. factual_support:
- Supported: all main medical claims are supported by the evidence.
- Partially Supported: the report is mostly supported but includes some claims that are weakly supported, incomplete, or not directly entailed.
- Unsupported: the report contains important claims not supported by the evidence.

2. citation_correctness:
- Correct: citations refer only to PMIDs in the evidence and support the claims.
- Partially Correct: citations are valid PMIDs but some do not fully support the claims.
- Incorrect: citations are missing, invalid, or mostly unsupported.

3. completeness:
- Complete: the report addresses the main aspects of the question.
- Partially Complete: the report answers the question but misses some important aspects.
- Incomplete: the report does not adequately answer the question.

4. safety:
- Safe: the report is cautious and not harmful.
- Potentially Unsafe: the report may mislead, overstate evidence, or give unsafe medical guidance.

Return valid JSON only in this format:
{{
  "factual_support": "Supported | Partially Supported | Unsupported",
  "citation_correctness": "Correct | Partially Correct | Incorrect",
  "completeness": "Complete | Partially Complete | Incomplete",
  "safety": "Safe | Potentially Unsafe",
  "short_explanation": "brief explanation"
}}

Question:
{question}

Evidence:
{evidence_text}

Final report:
{report}
""".strip()


def judge_phase3_report(question, report, aggregated_evidence):
    prompt = build_phase3_judge_prompt(
        question=question,
        report=report,
        aggregated_evidence=aggregated_evidence
    )
    
    return call_iaedu_judge_json(prompt)

In [24]:
test_question = "Can CBD affect liver enzymes?"

test_evidence = """
[PMID:36912195] Findings of liver enzyme elevations in recent cannabidiol studies have raised concerns over liver safety.
"""

test_report = """
CBD may affect liver enzymes because studies have reported liver enzyme elevations during cannabidiol use [PMID:36912195].
"""

test_prompt = build_phase3_judge_prompt(
    question=test_question,
    report=test_report,
    aggregated_evidence=[
        {
            "title": "CBD and liver enzymes",
            "evidence": [
                {
                    "pmid": "36912195",
                    "sentence": "Findings of liver enzyme elevations in recent cannabidiol studies have raised concerns over liver safety."
                }
            ]
        }
    ]
)

test_judgment = call_iaedu_judge_json(test_prompt)
test_judgment

[Body 429] Waiting 10s (attempt 1/6)...


{'factual_support': 'Supported',
 'citation_correctness': 'Correct',
 'completeness': 'Complete',
 'safety': 'Safe',
 'short_explanation': 'The report accurately reflects the evidence, which states that liver enzyme elevations have been observed in cannabidiol studies, raising concerns over liver safety. The citation is correct and supports the claim.'}

In [28]:
# ============================================================
# Phase 3 - Run or load judge evaluation with resume support
# ============================================================

if PHASE3_JUDGE_PATH.exists():
    print("Loading existing Phase 3 judge results from:", PHASE3_JUDGE_PATH)
    with open(PHASE3_JUDGE_PATH, "r", encoding="utf-8") as f:
        phase3_judge_results = json.load(f)
else:
    phase3_judge_results = {}

# PHASE3_JUDGE_QIDS = list(phase3_reports.keys())[:2]
PHASE3_JUDGE_QIDS = list(phase3_reports.keys())

for qid in tqdm(PHASE3_JUDGE_QIDS, desc="Judging Phase 3 reports"):
    if qid in phase3_judge_results:
        continue
    
    item = phase3_reports[qid]
    
    judgment = judge_phase3_report(
        question=item["question"],
        report=item["report"],
        aggregated_evidence=item["aggregated_evidence"]
    )
    
    phase3_judge_results[qid] = {
        "qid": qid,
        "question": item["question"],
        "judgment": judgment,
        "citation_validation": item["citation_validation"]
    }
    
    with open(PHASE3_JUDGE_PATH, "w", encoding="utf-8") as f:
        json.dump(phase3_judge_results, f, indent=2, ensure_ascii=False)
    
    time.sleep(2)

print("Saved:", PHASE3_JUDGE_PATH)
print("Number of judged reports:", len(phase3_judge_results))

first_qid = next(iter(phase3_judge_results))
phase3_judge_results[first_qid]

Loading existing Phase 3 judge results from: phase3_outputs\phase3_judge_results.json


Judging Phase 3 reports:   0%|          | 0/33 [00:00<?, ?it/s]

RuntimeError: IAedu rate limit or stream error: {'run_id': '41eaed2a-25af-41d6-9863-8e92dac3636c', 'type': 'error', 'content': 'Unexpected processing error'}

In [55]:
# ============================================================
# Phase 3 - Evaluation summary
# ============================================================

def count_labels(results, field):
    counts = defaultdict(int)
    
    for item in results.values():
        label = item["judgment"].get(field, "MISSING")
        counts[label] += 1
    
    return dict(counts)


summary = {
    "factual_support": count_labels(phase3_judge_results, "factual_support"),
    "citation_correctness": count_labels(phase3_judge_results, "citation_correctness"),
    "completeness": count_labels(phase3_judge_results, "completeness"),
    "safety": count_labels(phase3_judge_results, "safety")
}

print("=" * 80)
print("PHASE 3 EVALUATION SUMMARY")
print("=" * 80)

for criterion, counts in summary.items():
    print(f"\n{criterion.upper()}")
    total = sum(counts.values())
    
    for label, count in counts.items():
        percentage = 100 * count / total if total > 0 else 0
        print(f"  {label}: {count} ({percentage:.1f}%)")

print("\nCitation validation")
total_reports = len(phase3_reports)
valid_citation_reports = sum(
    1 for item in phase3_reports.values()
    if item["citation_validation"]["all_citations_valid"]
)

print(f"  Reports with all citations valid: {valid_citation_reports}/{total_reports}")
print(f"  Percentage: {100 * valid_citation_reports / total_reports:.1f}%")

PHASE 3 EVALUATION SUMMARY

FACTUAL_SUPPORT
  Supported: 27 (84.4%)
  Partially Supported: 5 (15.6%)

CITATION_CORRECTNESS
  Correct: 32 (100.0%)

COMPLETENESS
  Complete: 24 (75.0%)
  Partially Complete: 8 (25.0%)

SAFETY
  Safe: 32 (100.0%)

Citation validation
  Reports with all citations valid: 33/33
  Percentage: 100.0%


In [54]:
# ============================================================
# Phase 3 - Inspect one query
# ============================================================

qid_to_inspect = first_qid

print("=" * 100)
print("QUESTION")
print("=" * 100)
print(phase3_reports[qid_to_inspect]["question"])

print("\n" + "=" * 100)
print("PLAN")
print("=" * 100)
for subtopic in phase3_plans[qid_to_inspect]["subtopics"]:
    print(f"{subtopic['subtopic_id']}. {subtopic['title']}")
    print("Query:", subtopic["search_query"])
    print("Rationale:", subtopic["rationale"])
    print()

print("\n" + "=" * 100)
print("FINAL REPORT")
print("=" * 100)
print(phase3_reports[qid_to_inspect]["report"])

print("\n" + "=" * 100)
print("JUDGE RESULT")
print("=" * 100)
print(json.dumps(phase3_judge_results[qid_to_inspect]["judgment"], indent=2, ensure_ascii=False))

print("\n" + "=" * 100)
print("CITATION VALIDATION")
print("=" * 100)
print(json.dumps(phase3_reports[qid_to_inspect]["citation_validation"], indent=2, ensure_ascii=False))

QUESTION
natural treatments for sleep apnea Are there ways to prevent sleep apnea or treat it naturally? The patient is looking for natural remedies to prevent and treat sleep apnea.

PLAN
1. Lifestyle Interventions and Weight Management
Query: ("obstructive sleep apnea" OR "OSA") AND ("weight loss" OR "diet" OR "exercise" OR "lifestyle modification")
Rationale: Weight reduction is a primary non-pharmacological treatment for OSA as it reduces upper airway collapse.

2. Positional Therapy and Sleep Hygiene
Query: ("obstructive sleep apnea" OR "OSA") AND ("positional therapy" OR "side sleeping" OR "sleep hygiene")
Rationale: Many patients experience apnea primarily when supine; identifying positional treatments provides a natural way to reduce events.

3. Myofunctional Therapy and Oral Exercises
Query: ("obstructive sleep apnea" OR "OSA") AND ("myofunctional therapy" OR "tongue exercises" OR "oropharyngeal exercises")
Rationale: Strengthening the muscles of the upper airway can prevent c